# TUTORIAL: Real-time data assimilation on the Rijke tube


## Augmented state 

We perform state and parameter estimation in the RRijke tube model presented in [tutorial 05](05_Rijke_model.ipynb). 
The augmented state space vector of the Rijke tube model is
$$
\boldsymbol{\psi} = \begin{bmatrix}
        \boldsymbol{\phi}\\
        \boldsymbol{\alpha}\\
        \mathbf{p}_{mic}
        \end{bmatrix}
        =
        \begin{bmatrix}
        \boldsymbol{\eta} \\ 
        \boldsymbol{\mu} \\ 
        \boldsymbol{\nu} \\ 
        \beta\\
        \tau\\
        \mathbf{p}_{mic}
        \end{bmatrix}
        \in \mathbb{R}^{2N_m+N_c+2+N_q},
$$
where 
- $\boldsymbol{\eta}\in \mathbb{R}^{N_m}$: acoustic velocity modes (from Galerkin projection)
- $\boldsymbol{\mu}\in \mathbb{R}^{N_m}$: acoustic pressure modes (from Galerkin projection)
- $\boldsymbol{\nu}\in \mathbb{R}^{N_c}$: advection "velocity" modes (from Chevyshev projection)
- $\beta$: heat source strength
- $\tau$: acoustic time delay
- $\mathbf{p}_{mic} \mathbb{R}^{N_q}$: acoustic pressure at the microphone locations computed as
$$
{p}_{mic}(x_q, t) = -\sum^{N_m}_{j=1}\,\mu_j(t)\sin{\left(\dfrac{\omega_j}{\bar{c}} x_q\right)}. 
$$


In [ ]:
from observations import Observations
from models.physical import Rijke


truth = Observations(model=Rijke,
                     t_start=.2,
                     t_stop=.6,
                     Nt_obs=20,
                     beta=3.2,
                     tau=1.2e-3,
                     add_noise=True,
                     noise_type='pink, add',
                     noise_level=0.25,
                     )

truth.plot_truth(truth, f_max=2000, window=0.02, fig_width=12)

In [ ]:
from ensemble import Ensemble
from data_assimilation import EnKF  # try also EnSRKF


ensemble = Ensemble(parent_model=Rijke(dt=truth.dt),
                    da_method=EnKF,
                    m=10,
                    std_phi=0.25,
                    std_alpha=dict(beta=[3., 4.],
                                   tau=[1e-3, 2e-3]),
                    distribution_alpha='uniform',
                    inflation_factor=1.0,
                    inflation_factor_rejection=1.005,
                    )

ensemble.visualize_state()

In [ ]:
import numpy as np

filter_ens = ensemble.copy()

# Observation error covariance matrix
std_obs = 0.1
Cdd = np.diag(std_obs * np.ones(filter_ens.model.Nq)) * np.max(abs(truth.y_obs), axis=0) ** 2

# Assimilate all the observations sequentially: forecast to the observation time and analyse
for d, t_d in zip(truth.y_obs, truth.t_obs):
    filter_ens.forecast_step(t_end=t_d)
    filter_ens.analysis_step(d=d, Cdd=Cdd.copy())

# Forecast a bit further after the last observation and close the multiprocessing pools
filter_ens.forecast_step(t_end=truth.t_obs[-1] + 10 * filter_ens.model.t_CR, close=True)

In [ ]:
filter_ens.visualize_history(truth=truth, plot_members=True, dims=[0, 1])

In [ ]:
filter_ens.visualize_history(truth=truth, plot_members=False, reference_a=truth.true_parameters)